In [1]:
import numpy as np
import pandas as pd
from datetime import date, timedelta

In [2]:
np.random.seed(42)
N=10000

In [3]:
#Assinging attributes to each of 10,000 sessions

categories = ["Athleisure Apparel", "PerformanceFootwear", "Formal Footwear", "Fragrances", "Accessories"]
devices = ["Mobile", "Desktop", "Tablet"]
device_probs = [0.62, 0.30, 0.08]     #Example probabilities sum to 1
category_price_range = {
    "Athleisure Apparel": (399, 3999),
    "PerformanceFootwear": (1999, 7999),
    "Formal Footwear": (3999, 12999),
    "Fragrances": (399, 1999),
    "Accessories": (299, 2499),
}

user_id=[f"U{100000+i}" for i in range(N)]
category_probs = [0.38, 0.20, 0.12, 0.18, 0.12]
product_category = np.random.choice(categories,size=N,p=category_probs)
category_code = {
"Athleisure Apparel": "APP", "PerformanceFootwear": "PFT",
"Formal Footwear": "FFT", "Fragrances": "FRG", "Accessories": "ACC",
}
product_id = [f"P{np.random.randint(1000,1999)}_{category_code[cat]}"
for cat in product_category]
device = np.random.choice(devices, size=N,p=device_probs)
variant = np.random.choice(["A", "B"],size=N,p=[0.5, 0.5]) # 50/50split
price = [round(np.random.uniform(*category_price_range[cat]),-1)
for cat in product_category]
viewed = np.ones(N,dtype=int)

In [4]:
#Baseadd-to-cart probability by device
base_atc = {"Mobile": 0.24, "Desktop": 0.30, "Tablet": 0.27}

variant_b_lift = {"Mobile": 0.055, "Desktop": 0.025, "Tablet": 0.035}
atc_prob = np.array([
base_atc[d] + (variant_b_lift[d] if v == "B" else 0.0)
for d,v in zip(device,variant)
])
added_to_cart = np.random.binomial(1, atc_prob)

#Purchase,conditional onadd-to-cart—
base_purchase_given_atc = 0.42
variant_b_purchase_lift = 0.05
purchase_prob = np.where(
(added_to_cart == 1) & (variant == "B"),
base_purchase_given_atc + variant_b_purchase_lift,
np.where(added_to_cart == 1,base_purchase_given_atc, 0.0)
)
purchased = np.random.binomial(1,purchase_prob)
revenue = np.where(purchased == 1, price, 0)


In [5]:
session_date = date.today()
df = pd.DataFrame({
"user_id":user_id, "product_id": product_id,
"product_category":product_category, "activity": "product_page_view",
"price": price, "device":device, "variant": variant,
"session_date":session_date, "viewed":viewed,
"added_to_cart":added_to_cart, "purchased": purchased, "revenue": revenue
})
df.to_csv("one8_ab_test_sessions.csv", index=False)